# Parsing NCPDP pharmacy claims into a pandas DataFrame

**Goal:** read a synthetic **NCPDP Telecommunication (D.0-style)** file of pharmacy
claims, parse its **fixed header + field-ID segments**, merge each claim's **request
(billed)** with its **response (paid / rejected)**, and save a tidy CSV.

> **All data is fabricated. No PHI.** Amounts use implied 2-decimal cents (e.g. `4500`
> = \$45.00) without the overpunch sign of the real standard, and request/response records
> reuse one header layout — both simplified for teaching. See the notes at the end.

## What is NCPDP?

NCPDP — the National Council for Prescription Drug Programs — sets the standards that govern
how pharmacies and payers exchange prescription information in the United States. Its
**Telecommunication Standard** is the one behind almost every prescription fill: when a
pharmacist enters your prescription, the pharmacy system sends a real-time claim to the
pharmacy benefit manager (PBM) or payer and gets an answer back in seconds. That claim is
not X12 like a medical 837; instead NCPDP encodes each data element with a short
two-character **field ID** and separates fields, segments, and groups with non-printable
control characters. A single billing claim (transaction code **B1**) carries everything
needed to adjudicate the fill: the pharmacy and prescriber (by NPI), the member and plan,
and the drug itself — identified by its **NDC** — along with quantity dispensed, days
supply, and the pharmacy's submitted pricing.

Pharmacy billing is a two-message conversation. The pharmacy's request is answered by a
**B1 response** that either pays the claim — returning the ingredient cost paid, dispensing
fee, patient copay, and total amount paid — or rejects it with a reason code (for example,
drug not covered or prior authorization required). This billed-versus-paid pairing is the
pharmacy analog of the medical 837 → 835 cycle. NCPDP maintains several related standards
you may hear about too: **SCRIPT** for electronic prescribing (an XML format used between
prescribers and pharmacies), the **Batch Standard** for submitting many transactions at
once, and the **Post-Adjudication Standard**, a flat file of processed claims that is
usually what an analyst actually loads for reporting. A few practical notes: the NCPDP
standards are **proprietary and licensed** (so publicly shared examples, including this one,
are simplified), monetary fields use implied decimals with signed "overpunch" encoding, and
because claims contain protected health information they must be de-identified before use
outside of care.

**How an NCPDP transaction is built**

Unlike the `*`/`~` of X12, NCPDP uses non-printable delimiters and short **field IDs**:

| Delimiter | Byte | Role |
|-----------|------|------|
| Field Separator  | `0x1C` | precedes each field (`<sep>` + 2-char field ID + value) |
| Group Separator  | `0x1D` | splits the fixed header from the segment body |
| Segment Separator| `0x1E` | splits one segment from the next |

A **B1** transaction is a pharmacy billing claim; the payer's **B1 response** returns the
paid amounts (or a reject) — the pharmacy analog of the 837 → 835 pair.

## Setup

In [1]:
import urllib.request
from pathlib import Path
import pandas as pd

# NCPDP control-character delimiters
FS, GS, SS = "\x1c", "\x1d", "\x1e"

# Data source (read straight from GitHub raw). Swap for a local path
# (e.g. "ncpdp_pharmacy_claims.dat") to work offline.
SRC = "https://raw.githubusercontent.com/thousandoaks/Python4DS-II/refs/heads/main/datasets/ncpdp_pharmacy_claims.dat"
OUT = Path("ncpdp_claims_flattened.csv")

def load_text(src):
    src = str(src)
    if src.startswith(("http://", "https://")):
        with urllib.request.urlopen(src) as resp:
            return resp.read().decode("latin-1")
    return Path(src).read_text(encoding="latin-1")

raw = load_text(SRC)
records = [ln for ln in raw.split("\n") if ln.strip()]
print(len(records), "records")

# Show the first record with delimiters made visible
print(records[0].replace(FS, "<FS>").replace(GS, "<GS>").replace(SS, "<SS>"))

20 records
610415D0B1PHRXPLAN  1011234567890     20260701SYNTHV1   <GS><SS>AM04<FS>C2MBR200001<FS>C1RXGRPA<SS>AM01<FS>C419680312<FS>C52<SS>AM07<FS>EM1<FS>D27000001<FS>E103<FS>D790001000101<FS>E730<FS>D530<FS>D80<SS>AM03<FS>EZ01<FS>DB1987654321<SS>AM11<FS>D94500<FS>DC125<FS>DQ5000<FS>DU4625


## Reference: what the codes mean

These lookups turn cryptic segment and field IDs into readable names — handy for teaching
and for labeling the parsed output.

In [2]:
SEGMENT_NAMES = {
    "01": "Patient", "03": "Prescriber", "04": "Insurance", "07": "Claim", "11": "Pricing",
    "21": "Response Status", "22": "Response Claim", "23": "Response Pricing",
}
FIELD_NAMES = {
    "C1": "Group ID", "C2": "Cardholder ID", "C4": "Date of Birth", "C5": "Gender Code",
    "EM": "Rx/Svc Ref Qualifier", "D2": "Prescription Number", "E1": "Product ID Qualifier",
    "D7": "Product/Service ID (NDC)", "E7": "Quantity Dispensed", "D5": "Days Supply",
    "D8": "DAW Code", "EZ": "Prescriber ID Qualifier", "DB": "Prescriber ID",
    "D9": "Ingredient Cost Submitted", "DC": "Dispensing Fee Submitted",
    "DQ": "Usual & Customary", "DU": "Gross Amount Due",
    "AN": "Response Status", "F3": "Authorization Number", "FA": "Reject Count",
    "FB": "Reject Code", "F5": "Patient Pay Amount", "F6": "Ingredient Cost Paid",
    "F7": "Dispensing Fee Paid", "F9": "Total Amount Paid",
}

## Step 1 — Parse each record

The 56-byte **Transaction Header** is fixed-position; everything after the Group Separator
is a series of Segment-Separator-delimited segments, each starting with its 2-char code
(`AM` + code) followed by Field-Separator-delimited `<field-id><value>` pairs.

In [3]:
# Fixed Transaction Header layout: (name, start, end)
HEADER = [
    ("bin", 0, 6), ("version", 6, 8), ("txn_code", 8, 10), ("pcn", 10, 20),
    ("txn_count", 20, 21), ("provider_qual", 21, 23), ("provider_id", 23, 38),
    ("date_of_service", 38, 46), ("software_id", 46, 56),
]

def parse_record(line):
    head = {name: line[a:b].strip() for name, a, b in HEADER}
    body = line[56:]
    if body.startswith(GS):
        body = body[1:]
    segments = {}
    for raw_seg in body.split(SS):
        if not raw_seg:
            continue
        parts = raw_seg.split(FS)
        code = parts[0][2:]                       # strip leading "AM"
        segments[code] = {p[:2]: p[2:] for p in parts[1:] if p}
    return head, segments

# A record is a RESPONSE if it carries a Response Status segment (21), else a REQUEST
def is_response(segments):
    return "21" in segments

parsed = [parse_record(r) for r in records]
print("requests:", sum(not is_response(s) for _, s in parsed),
      "| responses:", sum(is_response(s) for _, s in parsed))

requests: 10 | responses: 10


## Step 2 — Build the billed (request) and paid (response) tables

In [4]:
def build_requests(parsed):
    rows = []
    for head, seg in parsed:
        if is_response(seg):
            continue
        clm, price = seg.get("07", {}), seg.get("11", {})
        ins, pat, presc = seg.get("04", {}), seg.get("01", {}), seg.get("03", {})
        rows.append({
            "rx_number": clm.get("D2"),
            "provider_npi": head["provider_id"],
            "date_of_service": head["date_of_service"],
            "cardholder_id": ins.get("C2"),
            "group_id": ins.get("C1"),
            "patient_dob": pat.get("C4"),
            "gender_code": pat.get("C5"),
            "prescriber_npi": presc.get("DB"),
            "ndc": clm.get("D7"),
            "quantity": clm.get("E7"),
            "days_supply": clm.get("D5"),
            "daw": clm.get("D8"),
            "ingredient_cost_submitted": price.get("D9"),
            "dispensing_fee_submitted": price.get("DC"),
            "usual_customary": price.get("DQ"),
            "gross_amount_due": price.get("DU"),
        })
    return pd.DataFrame(rows)

def build_responses(parsed):
    rows = []
    for head, seg in parsed:
        if not is_response(seg):
            continue
        status, rclaim, rprice = seg.get("21", {}), seg.get("22", {}), seg.get("23", {})
        rows.append({
            "rx_number": rclaim.get("D2"),
            "response_status": status.get("AN"),
            "reject_code": status.get("FB"),
            "auth_number": status.get("F3"),
            "ingredient_cost_paid": rprice.get("F6"),
            "dispensing_fee_paid": rprice.get("F7"),
            "patient_pay": rprice.get("F5"),
            "total_paid": rprice.get("F9"),
        })
    return pd.DataFrame(rows)

billed = build_requests(parsed)
paid = build_responses(parsed)
print(billed.shape, paid.shape)
billed.head()

(10, 16) (10, 8)


,rx_number,provider_npi,date_of_service,cardholder_id,group_id,patient_dob,gender_code,prescriber_npi,ndc,quantity,days_supply,daw,ingredient_cost_submitted,dispensing_fee_submitted,usual_customary,gross_amount_due
0,7000001,1234567890,20260701,MBR200001,RXGRPA,19680312,2,1987654321,90001000101,30,30,0,4500,125,5000,4625
1,7000002,1234567890,20260701,MBR200002,RXGRPA,19850522,1,1987654321,90002000101,30,30,0,1200,125,1500,1325
2,7000003,1234567890,20260702,MBR200003,RXGRPB,19590210,2,1900000032,90003000101,60,30,0,800,125,1200,925
3,7000004,1234567890,20260702,MBR200004,RXGRPB,19970801,1,1900000032,90004000101,30,30,0,1000,125,1300,1125
4,7000005,1234567890,20260703,MBR200005,RXGRPA,19740318,2,1987654321,90005000101,30,30,0,2200,125,2600,2325


## Step 3 — Merge billed + paid, decode values, convert money

Merge on the **prescription number**, translate the coded fields (gender, NDC → drug name,
reject code), and convert the implied-decimal cents to dollars.

In [5]:
merged = billed.merge(paid, on="rx_number", how="left")

# --- decode coded values ---
GENDER = {"1": "Male", "2": "Female", "0": "Unknown"}
DRUGS = {  # NDC -> drug (a tiny reference table; the claim carries only the NDC)
    "90001000101": "Atorvastatin 20 mg tab", "90002000101": "Lisinopril 10 mg tab",
    "90003000101": "Metformin 500 mg tab",   "90004000101": "Amlodipine 5 mg tab",
    "90005000101": "Omeprazole 20 mg cap",   "90006000101": "Levothyroxine 50 mcg tab",
    "90007000101": "Albuterol HFA inhaler",  "90008000101": "Sertraline 50 mg tab",
    "90009000101": "Amoxicillin 500 mg cap", "90010000101": "Gabapentin 300 mg cap",
}
REJECTS = {"70": "Product/Service Not Covered", "75": "Prior Authorization Required",
           "76": "Plan Limitations Exceeded", "88": "DUR Reject"}

merged["gender"] = merged["gender_code"].map(GENDER)
merged["drug_name"] = merged["ndc"].map(DRUGS)
merged["reject_reason"] = merged["reject_code"].map(REJECTS)
merged["date_of_service"] = pd.to_datetime(merged["date_of_service"], format="%Y%m%d")

# --- money: implied 2-decimal cents -> dollars ---
MONEY = ["ingredient_cost_submitted", "dispensing_fee_submitted", "usual_customary",
         "gross_amount_due", "ingredient_cost_paid", "dispensing_fee_paid",
         "patient_pay", "total_paid"]
for col in MONEY:
    merged[col] = pd.to_numeric(merged[col], errors="coerce") / 100.0

# --- reconcile paid claims: total_paid == ing_paid + disp_paid - patient_pay ---
paid_mask = merged["response_status"] == "P"
recon = (merged.loc[paid_mask, "total_paid"]
         - (merged.loc[paid_mask, "ingredient_cost_paid"]
            + merged.loc[paid_mask, "dispensing_fee_paid"]
            - merged.loc[paid_mask, "patient_pay"]))
print("paid claims reconcile:", bool(recon.abs().lt(0.005).all()))
print("rejected:", merged.loc[merged.response_status == "R",
                              ["rx_number", "drug_name", "reject_reason"]].to_dict("records"))
merged.head()

paid claims reconcile: True
rejected: [{'rx_number': '7000009', 'drug_name': 'Amoxicillin 500 mg cap', 'reject_reason': 'Product/Service Not Covered'}]


,rx_number,provider_npi,date_of_service,cardholder_id,group_id,patient_dob,gender_code,prescriber_npi,ndc,quantity,...,response_status,reject_code,auth_number,ingredient_cost_paid,dispensing_fee_paid,patient_pay,total_paid,gender,drug_name,reject_reason
0,7000001,1234567890,2026-07-01,MBR200001,RXGRPA,19680312,2,1987654321,90001000101,30,...,P,None,AUTH0001,38.0,1.0,10.0,29.0,Female,Atorvastatin 20 mg tab,NaN
1,7000002,1234567890,2026-07-01,MBR200002,RXGRPA,19850522,1,1987654321,90002000101,30,...,P,None,AUTH0002,9.0,1.0,5.0,5.0,Male,Lisinopril 10 mg tab,NaN
2,7000003,1234567890,2026-07-02,MBR200003,RXGRPB,19590210,2,1900000032,90003000101,60,...,P,None,AUTH0003,6.0,1.0,5.0,2.0,Female,Metformin 500 mg tab,NaN
3,7000004,1234567890,2026-07-02,MBR200004,RXGRPB,19970801,1,1900000032,90004000101,30,...,P,None,AUTH0004,7.5,1.0,5.0,3.5,Male,Amlodipine 5 mg tab,NaN
4,7000005,1234567890,2026-07-03,MBR200005,RXGRPA,19740318,2,1987654321,90005000101,30,...,P,None,AUTH0005,18.0,1.0,10.0,9.0,Female,Omeprazole 20 mg cap,NaN


## Step 4 — Flatten and save to CSV

In [6]:
col_order = [
    "rx_number", "response_status", "reject_reason", "date_of_service",
    "provider_npi", "prescriber_npi", "cardholder_id", "group_id",
    "patient_dob", "gender", "ndc", "drug_name", "quantity", "days_supply", "daw",
    "ingredient_cost_submitted", "dispensing_fee_submitted", "usual_customary",
    "gross_amount_due",
    "ingredient_cost_paid", "dispensing_fee_paid", "patient_pay", "total_paid",
    "auth_number",
]
flat = merged[col_order].sort_values("rx_number").reset_index(drop=True)
flat.to_csv(OUT, index=False)
print(f"Wrote {OUT.resolve()}  ({len(flat)} rows)")

print("\nTotal paid to pharmacy: $", round(flat["total_paid"].sum(), 2))
flat

Wrote /Users/davidlopez/Library/Application Support/Claude/local-agent-mode-sessions/edab4e8a-8f37-436c-81cb-ddb552643a37/63fcd7c8-ab49-48ff-9078-a2c9967287fc/local_17642962-1151-4965-a82e-13e83b434982/outputs/ncpdp_claims_flattened.csv  (10 rows)

Total paid to pharmacy: $ 115.5


,rx_number,response_status,reject_reason,date_of_service,provider_npi,prescriber_npi,cardholder_id,group_id,patient_dob,gender,...,daw,ingredient_cost_submitted,dispensing_fee_submitted,usual_customary,gross_amount_due,ingredient_cost_paid,dispensing_fee_paid,patient_pay,total_paid,auth_number
0,7000001,P,NaN,2026-07-01,1234567890,1987654321,MBR200001,RXGRPA,19680312,Female,...,0,45.0,1.25,50.00,46.25,38.0,1.0,10.0,29.0,AUTH0001
1,7000002,P,NaN,2026-07-01,1234567890,1987654321,MBR200002,RXGRPA,19850522,Male,...,0,12.0,1.25,15.00,13.25,9.0,1.0,5.0,5.0,AUTH0002
2,7000003,P,NaN,2026-07-02,1234567890,1900000032,MBR200003,RXGRPB,19590210,Female,...,0,8.0,1.25,12.00,9.25,6.0,1.0,5.0,2.0,AUTH0003
3,7000004,P,NaN,2026-07-02,1234567890,1900000032,MBR200004,RXGRPB,19970801,Male,...,0,10.0,1.25,13.00,11.25,7.5,1.0,5.0,3.5,AUTH0004
4,7000005,P,NaN,2026-07-03,1234567890,1987654321,MBR200005,RXGRPA,19740318,Female,...,0,22.0,1.25,26.00,23.25,18.0,1.0,10.0,9.0,AUTH0005
5,7000006,P,NaN,2026-07-03,1234567890,1900000064,MBR200006,RXGRPC,19530909,Male,...,0,15.0,1.25,18.00,16.25,11.0,1.0,5.0,7.0,AUTH0006
6,7000007,P,NaN,2026-07-04,1234567890,1900000064,MBR200007,RXGRPC,19911125,Female,...,0,65.0,1.25,75.00,66.25,58.0,1.0,15.0,44.0,AUTH0007
7,7000008,P,NaN,2026-07-04,1234567890,1900000032,MBR200008,RXGRPB,19660704,Male,...,0,18.0,1.25,21.00,19.25,14.0,1.0,10.0,5.0,AUTH0008
8,7000009,R,Product/Service Not Covered,2026-07-05,1234567890,1987654321,MBR200009,RXGRPA,19801230,Female,...,0,15.0,1.25,16.25,16.25,NaN,NaN,NaN,NaN,None
9,7000010,P,NaN,2026-07-05,1234567890,1900000064,MBR200010,RXGRPC,19450615,Male,...,0,24.0,1.25,28.00,25.25,20.0,1.0,10.0,11.0,AUTH0010


### Notes for the seminar

- NCPDP identifies every element with a short **field ID** (`D7` = NDC, `F9` = Total Amount
  Paid); parsing is "split on the delimiters, then look up the IDs."
- The claim carries the drug only as an **NDC** — the name comes from a **join** to a
  reference table (shown here with a small dict).
- The **request** is what the pharmacy billed; the **response** is what the plan paid or
  why it rejected — the same billed-vs-paid story as 837 → 835.
- **Real-world caveats:** the NCPDP Telecommunication standard is **proprietary/licensed**;
  production money fields use implied decimals **with overpunch signs**; and real
  transactions are **real-time** messages (this file bundles them for a self-contained
  example). Headers also carry **PHI** and would need de-identification before sharing.